# Fraud Detection System
This project implements a fraud detection system using the Credit Card Fraud Detection dataset. The dataset contains anonymized financial transaction data, with each transaction labeled as either fraudulent (1) or legitimate (0). Due to the severe imbalance between fraudulent and legitimate transactions, the project focuses not only on building predictive models but also on proper data preprocessing and evaluation to handle class imbalance effectively. The end goal is to create a Python-based system that accurately detects fraudulent transactions and includes an interactive testing interface

## Importing Libraries

In [69]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
import pickle
import os
import joblib

## Loading and Previewing Dataset

In [70]:
df= pd.read_csv('creditcard.csv')

In [71]:
print(df.head())

   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

In [72]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

In [73]:
print(df.isnull().sum)

<bound method NDFrame._add_numeric_operations.<locals>.sum of          Time     V1     V2     V3     V4     V5     V6     V7     V8     V9  \
0       False  False  False  False  False  False  False  False  False  False   
1       False  False  False  False  False  False  False  False  False  False   
2       False  False  False  False  False  False  False  False  False  False   
3       False  False  False  False  False  False  False  False  False  False   
4       False  False  False  False  False  False  False  False  False  False   
...       ...    ...    ...    ...    ...    ...    ...    ...    ...    ...   
284802  False  False  False  False  False  False  False  False  False  False   
284803  False  False  False  False  False  False  False  False  False  False   
284804  False  False  False  False  False  False  False  False  False  False   
284805  False  False  False  False  False  False  False  False  False  False   
284806  False  False  False  False  False  False  False  F

## Exploring Data Imbalance
- Examine the distribution of classes
- Reveals extreme imbalance (492 fraud cases vs 284,315 legitimate transactions)

In [74]:
df['Class'].value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

## Data Separation and Analysis
- Separate data into legitimate (normal) and fraudulent transactions
- Compare statistical measures between the two classes
- Fraud transactions tend to have higher average amounts
- Mean values of V1-V28 features differ significantly between classes

In [75]:
# Data is highly imbalance
# Seprating data for balancing
normal= df[df.Class==0]
fraud= df[df.Class==1]

In [76]:
print(normal.shape)
print(fraud.shape)

(284315, 31)
(492, 31)


In [77]:
# Statistical measures
normal.Amount.describe()

count    284315.000000
mean         88.291022
std         250.105092
min           0.000000
25%           5.650000
50%          22.000000
75%          77.050000
max       25691.160000
Name: Amount, dtype: float64

In [78]:
fraud.Amount.describe()

count     492.000000
mean      122.211321
std       256.683288
min         0.000000
25%         1.000000
50%         9.250000
75%       105.890000
max      2125.870000
Name: Amount, dtype: float64

In [79]:
# Comparing the value for both transaction classes
df.groupby('Class').mean()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
Class,,,,,,,,,,,,,,,,,,,,,
0,94838.202258,0.008258,-0.006271,0.012171,-0.007860,0.005453,0.002419,0.009637,-0.000987,0.004467,...,-0.000644,-0.001235,-0.000024,0.000070,0.000182,-0.000072,-0.000089,-0.000295,-0.000131,88.291022
1,80746.806911,-4.771948,3.623778,-7.033281,4.542029,-3.151225,-1.397737,-5.568731,0.570636,-2.581123,...,0.372319,0.713588,0.014049,-0.040308,-0.105130,0.041449,0.051648,0.170575,0.075667,122.211321


## Handling Class Imbalance (Undersampling)
- Create a balanced dataset by randomly selecting 492 legitimate transactions (same number as fraud cases)
- Combine with all fraud cases to create a new balanced dataset
- Results in 492 legitimate and 492 fraud cases (balanced)

In [80]:
# Under_Sampling
normal_sample= normal.sample(n=492)

In [81]:
new_dataset= pd.concat([normal_sample, fraud], axis=0)

In [82]:
print(new_dataset.head())

            Time        V1        V2        V3        V4        V5        V6  \
32263    36714.0  1.233231  0.107071  0.251467  1.093559  0.129778  0.556558   
243818  152087.0  2.033815  0.621054 -2.485445  0.558556  0.890040 -1.234270   
52455    45460.0 -0.925654 -0.125770  2.315891 -0.228360 -0.389407  1.041508   
263440  160955.0  2.004927 -0.168802 -1.027683  0.320840 -0.172520 -0.873308   
55239    46844.0 -1.046613  0.751170  2.046745  1.387884 -0.976121  0.479539   

              V7        V8        V9  ...       V21       V22       V23  \
32263  -0.192972  0.131512  0.290480  ... -0.101927 -0.047213 -0.261886   
243818  0.351750 -0.312150  0.081709  ...  0.138210  0.630405 -0.123538   
52455   2.206091 -2.043776  0.157195  ... -0.718135 -0.037870 -0.384860   
263440  0.009342 -0.117965  0.406110  ... -0.236434 -0.618338  0.355789   
55239  -0.514190  0.917658  0.315330  ...  0.045680  0.163561  0.002180   

             V24       V25       V26       V27       V28  Amount  Cl

In [83]:
new_dataset['Class'].value_counts()

Class
0    492
1    492
Name: count, dtype: int64

In [84]:
# Split data into training sets
X = new_dataset.drop("Class", axis=1)
y = new_dataset["Class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Model Training (Random Forest)
- Initialize a Random Forest classifier with 100 trees
- Train the model on the balanced training data
#### Random Forest is chosen for its:
- Ability to handle high-dimensional data
- Feature importance calculation
- Good performance on imbalanced data (when balanced)

In [85]:
# Model Training - Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

## Model Evaluation
To evaluate model performance, key metrics such as precision, recall, and F1-score are calculated using the test set. These metrics are especially important in fraud detection:

- Precision tells us the percentage of predicted frauds that were actually frauds.
- Recall indicates the percentage of actual frauds that were correctly detected.
- F1-score balances precision and recall into a single metric.

In [86]:
# Evaluate the model
y_pred = model.predict(X_test)
print("Model Evaluation Report:")
print(classification_report(y_test, y_pred))

Model Evaluation Report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.94        99
           1       1.00      0.87      0.93        98

    accuracy                           0.93       197
   macro avg       0.94      0.93      0.93       197
weighted avg       0.94      0.93      0.93       197



## Model Saving
- Save trained model to disk for future use
- Allows loading the model without retraining

In [87]:
# Save model
joblib.dump(model, 'fraud_detection_model.pkl')

['fraud_detection_model.pkl']

## Prediction Interface
### Purpose:
- Load saved model
- Create interactive interface for users to input transaction details
- Make real-time predictions on new data
- Output clear fraud/legitimate classification

In [92]:
# Load trained model
model = joblib.load('fraud_detection_model.pkl')

# Get feature names used during training
feature_names = model.feature_names_in_

print("Enter values for each of the following features:")

# Get user input for each feature
input_values = []
for feature in feature_names:
    while True:
        try:
            value = float(input(f"{feature}: "))
            input_values.append(value)
            break
        except ValueError:
            print("Please enter a valid numeric value.")

# Convert input to DataFrame with correct column names
user_input_df = pd.DataFrame([input_values], columns=feature_names)

# Make prediction
prediction = model.predict(user_input_df)

# Show result
if prediction[0] == 1:
    print("\n⚠️ Fraudulent Transaction Detected!")
else:
    print("\n✅ Legitimate Transaction.")


Enter values for each of the following features:


Time:  0
V1:  1.1919
V2:  0.2662
V3:  0.1665
V4:  0.4482
V5:  0.06
V6:  -0.082
V7:  -0.079
V8:  0.0851
V9:  -0.255
V10:  -0.167
V11:  1.6127
V12:  1.0652
V13:  0.4891
V14:  -0.144
V15:  0.6356
V16:  0.4639
V17:  -0.115
V18:  -0.183
V19:  -0.146
V20:  -0.069
V21:  -0.226
V22:  -0.639
V23:  0.1013
V24:  -0.34
V25:  0.1672
V26:  0.1259
V27:  -0.009
V28:  0.0147
Amount:  2.69



✅ Legitimate Transaction.


## Key Considerations
- Data Imbalance Handling: The project uses undersampling to balance classes, which is simple but may discard potentially useful data. Alternatives like SMOTE or class weights could be considered.

- Feature Engineering: All original features are used. Domain-specific feature engineering might improve performance.

- Model Selection: Random Forest performs well here, but other algorithms like Logistic Regression or Gradient Boosting could be compared.

- Evaluation Metrics: In fraud detection, recall (identifying all frauds) is often more important than precision, as false negatives are costlier than false positives.

- Scalability: The current implementation is suitable for batch processing. For real-time systems, consider:

- Model deployment as a web service

- Automated feature extraction pipelines

- Continuous model monitoring and retraining

